In [1]:
from datetime import datetime, timezone
from pathlib import Path

import catboost
import joblib
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

In [2]:
MODEL_VERSION = "1.0.0"
RANDOM_STATE = 42
OUT_PATH = Path("../models/credit_model.joblib")

In [3]:
data = fetch_openml("credit-g", version=1, as_frame=True)
df = data.frame.copy()

In [4]:
TARGET = "class"
y = (df[TARGET] == "bad").astype(int)
X = df.drop(columns=[TARGET])

CAT_FEATURES = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
NUM_FEATURES = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
FEATURES = CAT_FEATURES + NUM_FEATURES  # порядок, в котором пайплайн ждёт колонки

# категории -> обычные строки (сервис тоже будет присылать строки)
X[CAT_FEATURES] = X[CAT_FEATURES].astype(str)
X[NUM_FEATURES] = X[NUM_FEATURES].astype(float)
X = X[FEATURES]

print(X.shape, "доля bad:", round(y.mean(), 3))
print("cat:", CAT_FEATURES)
print("num:", NUM_FEATURES)

(1000, 20) доля bad: 0.3
cat: ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']
num: ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [6]:
def make_pipeline() -> Pipeline:
    prep = ColumnTransformer(
        [
            ("cat", SimpleImputer(strategy="constant", fill_value="missing"), CAT_FEATURES),
            ("num", SimpleImputer(strategy="median"), NUM_FEATURES),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=4,
        cat_features=CAT_FEATURES,
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,  # не создавать catboost_info/
        thread_count=1,             # детерминизм + предсказуемое потребление CPU в k8s
    )
    return Pipeline([("prep", prep), ("model", model)])

In [7]:
COST_FN, COST_FP = 5, 1  # FN: пропустили "bad", FP: отказали "good"

# cross_val_predict не используем: он клонирует модель через sklearn.clone,
# а CatBoost с cat_features=list клонирование не переживает. Свой цикл проще.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(X_train))
for tr_idx, va_idx in cv.split(X_train, y_train):
    fold = make_pipeline().fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    oof[va_idx] = fold.predict_proba(X_train.iloc[va_idx])[:, 1]

def cost(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    fn = ((pred == 0) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    return COST_FN * fn + COST_FP * fp

grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
costs = [cost(y_train.values, oof, t) for t in grid]
THRESHOLD = float(grid[int(np.argmin(costs))])
print("OOF ROC AUC:", round(roc_auc_score(y_train, oof), 3), "| порог:", THRESHOLD)


OOF ROC AUC: 0.8 | порог: 0.16


In [8]:
pipeline = make_pipeline().fit(X_train, y_train)

proba_test = pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, proba_test)
test_cost = cost(y_test.values, proba_test, THRESHOLD)
print("test ROC AUC:", round(test_auc, 3), "| test cost:", test_cost)

test ROC AUC: 0.805 | test cost: 113


In [9]:
feature_values = {
    **{c: sorted(X[c].unique().tolist()) for c in CAT_FEATURES},
    **{c: {"min": float(X[c].min()), "max": float(X[c].max())} for c in NUM_FEATURES},
}

metadata = {
    "model_name": "german-credit-catboost",
    "version": MODEL_VERSION,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "features": FEATURES,            # порядок колонок для DataFrame в сервисе
    "cat_features": CAT_FEATURES,
    "num_features": NUM_FEATURES,
    "feature_values": feature_values,  # допустимые категории и диапазоны: подсказка для схемы запроса
    "target": "bad (1) vs good (0)",
    "threshold": THRESHOLD,          # decision = proba_bad >= threshold -> отказ
    "metrics": {"test_roc_auc": round(float(test_auc), 4), "oof_roc_auc": round(float(roc_auc_score(y_train, oof)), 4)},
    "libs": {"scikit-learn": sklearn.__version__, "catboost": catboost.__version__},
}

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipeline, "metadata": metadata}, OUT_PATH)
print("saved:", OUT_PATH)

saved: ../models/credit_model.joblib
